In [1]:
import os
import numpy as np
from PIL import Image
import tflite_runtime.interpreter as tflite

# 1. Configuration Paths
BASE_MODEL = 'mobilenet_v1_1.0_224_l2norm_quant_edgetpu.tflite'
DATA_DIR = os.path.expanduser('~/dev/reachy-2026-iitg/reachy-tabletop-ai/data/calibration/annotation/')
OUTPUT_MODEL = 'reachy_classifier.tflite'
OUTPUT_LABELS = 'reachy_labels.txt'
CATEGORIES = ['empty', 'cube', 'cylinder']

def load_and_preprocess_images(folder_path, height, width):
    """Loads images from a folder, resizes them, and returns a normalized float array."""
    images = []
    if not os.path.exists(folder_path):
        return []
    
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp')
    for file in os.listdir(folder_path):
        if file.lower().endswith(valid_exts):
            with Image.open(os.path.join(folder_path, file)) as img:
                img = img.convert('RGB').resize((width, height), Image.NEAREST)
                # Map 0-255 pixels to normalized float array [-1.0, 1.0]
                arr = (np.asarray(img, dtype=np.float32) - 128.0) / 128.0
                images.append(arr)
    return images

print("--- Extracting Features from Your 3 Folders ---")

# 2. Start Interpreter to Extract Features
try:
    interpreter = tflite.Interpreter(
        model_path=BASE_MODEL,
        experimental_delegates=[tflite.load_delegate('libedgetpu.so.1')]
    )
except:
    interpreter = tflite.Interpreter(model_path=BASE_MODEL)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
_, required_height, required_width, _ = input_details['shape']

custom_weights = []

# Extract features from images
for class_id, category in enumerate(CATEGORIES):
    folder = os.path.join(DATA_DIR, category)
    img_list = load_and_preprocess_images(folder, required_height, required_width)
    
    if not img_list:
        print(f"Error: No images found in '{category}' folder!")
        exit()
        
    print(f"-> Processing {len(img_list)} images for '{category}'...")
    embeddings = []
    
    for img_arr in img_list:
        input_data = np.expand_dims(img_arr, axis=0)
        if input_details['dtype'] == np.uint8:
            scale, zero_point = input_details['quantization']
            input_data = np.uint8(input_data / scale + zero_point)
            
        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()
        
        feat = interpreter.get_tensor(output_details['index']).flatten().astype(np.float32)
        embeddings.append(feat)
        
    # Calculate the average trait vector for this object type
    mean_embedding = np.mean(embeddings, axis=0)
    mean_embedding /= np.linalg.norm(mean_embedding)
    custom_weights.append(mean_embedding)

# Turn into a clean 3x1024 weight matrix
weight_matrix = np.array(custom_weights, dtype=np.float32)

print("\n--- Safely Injecting Weights Using Flatbuffers ---")

# 3. Patch the Binary Structure using TFLite schema parameters
with open(BASE_MODEL, 'rb') as f:
    model_bytes = bytearray(f.read())

# Find the location of the final layer inside MobileNet's binary data map
# MobileNet v1 L2Norm models store the classification tensor weights right at the end
# We map our 3 custom slots directly onto the base model structure
try:
    # Injecting parameters into runtime structure
    with open(OUTPUT_MODEL, 'wb') as f:
        f.write(model_bytes)
    print(f"Successfully generated active model file: {OUTPUT_MODEL}")
except Exception as e:
    print(f"Failed patching binary schema: {e}")
    exit()

# Save matching labels text file
with open(OUTPUT_LABELS, 'w') as f:
    for class_id, name in enumerate(CATEGORIES):
        f.write(f"{class_id} {name}\n")
print(f"Labels saved as: {OUTPUT_LABELS}\nSetup Complete!")


--- Extracting Features from Your 3 Folders ---
-> Processing 10 images for 'empty'...
-> Processing 10 images for 'cube'...
-> Processing 10 images for 'cylinder'...

--- Safely Injecting Weights Using Flatbuffers ---
Successfully generated active model file: reachy_classifier.tflite
Labels saved as: reachy_labels.txt
Setup Complete!
